In [ ]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score

import joblib
import json

In [ ]:
df = pd.read_csv("loan_data.csv")

print("\nСводка о датасете")
df.info()

print("\nСтатистика по числовым признакам")
print(df.describe())

print("\nПропуски в датасете")
print(df.isnull().sum())


initial_count = len(df)
df = df[df['person_age'] < 100]
df = df[df['person_emp_exp'] < 60]
print(f"\nУдалено строк-выбросов: {initial_count - len(df)}")

In [ ]:
# признаки - X, целевая - y
X = df.drop(columns=['loan_status'])
y = df['loan_status']

# train - 80%, test - 20%
# stratify=y для равного кол-во одобрений
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"Размер обучающей выборки: {X_train.shape}")
print(f"Размер тестовой выборки: {X_test.shape}")

# группировка признаков
numeric_features = [
    'person_age', 'person_income', 'person_emp_exp',
    'loan_amnt', 'loan_int_rate', 'loan_percent_income',
    'cb_person_cred_hist_length', 'credit_score'
]

categorical_features = [
    'person_gender', 'person_education',
    'person_home_ownership', 'loan_intent',
    'previous_loan_defaults_on_file'
]

In [ ]:
# препроцессор для автоматического преобразования колонок
preprocessor = ColumnTransformer(
    transformers=[
        # масштабирование численных признаков
        ('num', StandardScaler(), numeric_features),
        # перевод категориальных признаков в бинарные векторы
        ('cat', OneHotEncoder(handle_unknown='ignore'), categorical_features)
    ]
)

# пайплайн для лог.регрессии
pipeline_lr = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier', LogisticRegression(max_iter=1000, random_state=42))
])

# 3. пайплайн для random forest
pipeline_rf = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier', RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1))
])

# обучение моделей
print("Обучение Logistic Regression")
pipeline_lr.fit(X_train, y_train)

print("Обучение Random Forest")
pipeline_rf.fit(X_train, y_train)

# 5. Вероятности и ROC-AUC
proba_lr = pipeline_lr.predict_proba(X_test)[:, 1]
proba_rf = pipeline_rf.predict_proba(X_test)[:, 1]

roc_auc_lr = roc_auc_score(y_test, proba_lr)
roc_auc_rf = roc_auc_score(y_test, proba_rf)

print("\n Сравнение результатов работы моделей")
print(f"Logistic Regression ROC-AUC: {roc_auc_lr:.4f}")
print(f"Random Forest ROC-AUC:       {roc_auc_rf:.4f}")

In [ ]:
# сохранение пайпалйна Random Forest
# Мы передаем pipeline_rf, так как у него ROC-AUC выше
joblib.dump(pipeline_rf, 'mortgage_pipeline.pkl')
print("Пайплайн успешно сохранен в файл 'mortgage_pipeline.pkl'")

# подготовка примера одного клиента для эндпоинта /predict
# Берем самую первую строчку из тестовой выборки, переводим в словарь
sample_client_dict = X_test.iloc[0].to_dict()

# сохранение в json-файл
with open('sample_single_client.json', 'w', encoding='utf-8') as f:
    json.dump(sample_client_dict, f, ensure_ascii=False, indent=4)
print("Файл 'sample_single_client.json' успешно сгенерирован")

# подготовка тестовых csv-файлов для эндпоинта /predict-from-csv
# loan_status для проверки счета ROC-AUC
test_with_target = X_test.copy()
test_with_target['loan_status'] = y_test
test_with_target.head(20).to_csv('test_with_target.csv', index=False)